# 01 · Limpieza de datos — FilmAnalytics

**Objetivo:** preparar un dataset confiable para predecir si una película será un éxito comercial.

- **Entrada:** `data/raw/TMDB_movie_dataset.csv` (dataset completo de TMDB)
- **Salida:** `data/movies_clean.csv`, que usa el notebook `02_modelo.ipynb`

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
data_movies = pd.read_csv("../data/raw/TMDB_movie_dataset.csv")
embudo = [("Dataset original de TMDB", len(data_movies))]
data_movies.head()

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,original_language,original_title,overview,popularity,poster_path,tagline,genres,production_companies,production_countries,spoken_languages
0,27205,Inception,8.364,34495,Released,2010-07-15,825532764,148,False,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,...,en,Inception,"Cobb, a skilled thief who commits corporate es...",83.952,/oYuLEt3zVCKq57qu2F8dT7NIa6f.jpg,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili"
1,157336,Interstellar,8.417,32571,Released,2014-11-05,701729206,169,False,/pbrkL804c8yAv3zBZR4QPEafpAR.jpg,...,en,Interstellar,The adventures of a group of explorers who mak...,140.241,/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg,Mankind was born on Earth. It was never meant ...,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",English
2,155,The Dark Knight,8.512,30619,Released,2008-07-16,1004558444,152,False,/nMKdUUepR0i5zn0y1T4CsSB5chy.jpg,...,en,The Dark Knight,Batman raises the stakes in his war on crime. ...,130.643,/qJ2tW6WMUDux911r6m7haRef0WH.jpg,Welcome to a world without rules.,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America","English, Mandarin"
3,19995,Avatar,7.573,29815,Released,2009-12-15,2923706026,162,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,...,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",79.932,/kyeqWdyUXW608qlYkRqosgbbJyK.jpg,Enter the world of Pandora.,"Action, Adventure, Fantasy, Science Fiction","Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom","English, Spanish"
4,24428,The Avengers,7.710,29166,Released,2012-04-25,1518815515,143,False,/9BBTo63ANSmhC4e6r62OJFuK2GL.jpg,...,en,The Avengers,When an unexpected enemy emerges and threatens...,98.082,/RYMX2wcKCBAr24UyPD7xwmjaTn.jpg,Some assembly required.,"Science Fiction, Action, Adventure",Marvel Studios,United States of America,"English, Hindi, Russian"


 ## 1. Selección de columnas
  Elimino columnas que no aportan a la predicción: texto libre (sinopsis, tagline), rutas de imágenes,
  identificadores externos y `runtime`.

In [3]:
data_movies = data_movies.drop(columns = ["original_title", "adult", "backdrop_path", "homepage", "poster_path", "overview", "tagline", "spoken_languages", "status", "imdb_id", "runtime"])

In [4]:
data_movies.head()

,id,title,vote_average,vote_count,release_date,revenue,budget,original_language,popularity,genres,production_companies,production_countries
0,27205,Inception,8.364,34495,2010-07-15,825532764,160000000,en,83.952,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America"
1,157336,Interstellar,8.417,32571,2014-11-05,701729206,165000000,en,140.241,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America"
2,155,The Dark Knight,8.512,30619,2008-07-16,1004558444,185000000,en,130.643,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America"
3,19995,Avatar,7.573,29815,2009-12-15,2923706026,237000000,en,79.932,"Action, Adventure, Fantasy, Science Fiction","Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom"
4,24428,The Avengers,7.710,29166,2012-04-25,1518815515,220000000,en,98.082,"Science Fiction, Action, Adventure",Marvel Studios,United States of America


## 2. Filtro de confiabilidad
Conservo solo películas con **al menos 100 votos**. Con menos votos los datos suelen estar
incompletos y las métricas son poco representativas.

In [5]:
data_movies = data_movies[data_movies["vote_count"] >= 100]
embudo.append(("Con al menos 100 votos", len(data_movies)))
data_movies.head()

,id,title,vote_average,vote_count,release_date,revenue,budget,original_language,popularity,genres,production_companies,production_countries
0,27205,Inception,8.364,34495,2010-07-15,825532764,160000000,en,83.952,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America"
1,157336,Interstellar,8.417,32571,2014-11-05,701729206,165000000,en,140.241,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America"
2,155,The Dark Knight,8.512,30619,2008-07-16,1004558444,185000000,en,130.643,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America"
3,19995,Avatar,7.573,29815,2009-12-15,2923706026,237000000,en,79.932,"Action, Adventure, Fantasy, Science Fiction","Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom"
4,24428,The Avengers,7.710,29166,2012-04-25,1518815515,220000000,en,98.082,"Science Fiction, Action, Adventure",Marvel Studios,United States of America


## 3. Año de estreno
Convierto `release_date` a fecha y extraigo el año. Elimino registros sin fecha válida o anteriores a 1900.

In [6]:
data_movies['release_date'] = pd.to_datetime(data_movies['release_date'], errors='coerce')
data_movies['year'] = data_movies['release_date'].dt.year

In [7]:
data_movies.head()

,id,title,vote_average,vote_count,release_date,revenue,budget,original_language,popularity,genres,production_companies,production_countries,year
0,27205,Inception,8.364,34495,2010-07-15,825532764,160000000,en,83.952,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America",2010.0
1,157336,Interstellar,8.417,32571,2014-11-05,701729206,165000000,en,140.241,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",2014.0
2,155,The Dark Knight,8.512,30619,2008-07-16,1004558444,185000000,en,130.643,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America",2008.0
3,19995,Avatar,7.573,29815,2009-12-15,2923706026,237000000,en,79.932,"Action, Adventure, Fantasy, Science Fiction","Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom",2009.0
4,24428,The Avengers,7.710,29166,2012-04-25,1518815515,220000000,en,98.082,"Science Fiction, Action, Adventure",Marvel Studios,United States of America,2012.0


In [8]:
data_movies = data_movies.dropna(subset=['year'])
data_movies = data_movies.drop(columns=['release_date'])
data_movies['year'] = data_movies['year'].astype(int)
data_movies.head()

,id,title,vote_average,vote_count,revenue,budget,original_language,popularity,genres,production_companies,production_countries,year
0,27205,Inception,8.364,34495,825532764,160000000,en,83.952,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America",2010
1,157336,Interstellar,8.417,32571,701729206,165000000,en,140.241,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",2014
2,155,The Dark Knight,8.512,30619,1004558444,185000000,en,130.643,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America",2008
3,19995,Avatar,7.573,29815,2923706026,237000000,en,79.932,"Action, Adventure, Fantasy, Science Fiction","Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom",2009
4,24428,The Avengers,7.710,29166,1518815515,220000000,en,98.082,"Science Fiction, Action, Adventure",Marvel Studios,United States of America,2012


In [9]:
data_movies = data_movies[data_movies["year"] >= 1900]
embudo.append(("Con año válido (≥ 1900)", len(data_movies)))
data_movies.head()

,id,title,vote_average,vote_count,revenue,budget,original_language,popularity,genres,production_companies,production_countries,year
0,27205,Inception,8.364,34495,825532764,160000000,en,83.952,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America",2010
1,157336,Interstellar,8.417,32571,701729206,165000000,en,140.241,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",2014
2,155,The Dark Knight,8.512,30619,1004558444,185000000,en,130.643,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America",2008
3,19995,Avatar,7.573,29815,2923706026,237000000,en,79.932,"Action, Adventure, Fantasy, Science Fiction","Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom",2009
4,24428,The Avengers,7.710,29166,1518815515,220000000,en,98.082,"Science Fiction, Action, Adventure",Marvel Studios,United States of America,2012


## 4. Género y país principal
Una película puede tener varios géneros y países. Me quedo con el **primero de cada lista** como
el principal, para tener una sola categoría por película.

In [10]:
data_movies = data_movies.copy()
data_movies['genre_main'] = data_movies['genres'].apply(lambda x: x.split(',')[0].strip() if pd.notnull(x) else None)

In [11]:
data_movies['country_main'] = data_movies['production_countries'].apply(lambda x: x.split(',')[0].strip() if pd.notnull(x) else None)

In [12]:
data_movies = data_movies.drop(columns=['genres', 'production_countries'])

In [13]:
data_movies.head()

,id,title,vote_average,vote_count,revenue,budget,original_language,popularity,production_companies,year,genre_main,country_main
0,27205,Inception,8.364,34495,825532764,160000000,en,83.952,"Legendary Pictures, Syncopy, Warner Bros. Pict...",2010,Action,United Kingdom
1,157336,Interstellar,8.417,32571,701729206,165000000,en,140.241,"Legendary Pictures, Syncopy, Lynda Obst Produc...",2014,Adventure,United Kingdom
2,155,The Dark Knight,8.512,30619,1004558444,185000000,en,130.643,"DC Comics, Legendary Pictures, Syncopy, Isobel...",2008,Drama,United Kingdom
3,19995,Avatar,7.573,29815,2923706026,237000000,en,79.932,"Dune Entertainment, Lightstorm Entertainment, ...",2009,Action,United States of America
4,24428,The Avengers,7.710,29166,1518815515,220000000,en,98.082,Marvel Studios,2012,Science Fiction,United States of America


## 5. Ingresos y presupuesto
En TMDB, un valor de 0 casi siempre significa "dato no registrado", no que la película costó o ganó $0.
Como el objetivo depende del ingreso, elimino las películas sin ingreso o sin presupuesto.

In [14]:
print((data_movies["revenue"]==0).sum())
print((data_movies["budget"]==0).sum())

8895
9491


In [15]:
print("Registros actuales:", len(data_movies))
print("Con revenue > 0:", (data_movies['revenue'] > 0).sum())
print("Con revenue > 0 Y budget > 0:", ((data_movies['revenue'] > 0) & (data_movies['budget'] > 0)).sum())

Registros actuales: 18177
Con revenue > 0: 9282
Con revenue > 0 Y budget > 0: 7070


In [16]:
data_movies = data_movies[(data_movies['revenue'] > 0) & (data_movies['budget'] > 0)]
embudo.append(("Con ingreso y presupuesto > 0", len(data_movies)))
print("Registros finales:", len(data_movies))

Registros finales: 7070


In [17]:
print(data_movies['genre_main'].value_counts().head(20))

genre_main
Drama              1571
Comedy             1464
Action             1078
Horror              536
Adventure           456
Crime               348
Thriller            328
Animation           234
Science Fiction     195
Fantasy             177
Romance             167
Family              159
Mystery              97
War                  67
Western              59
Music                57
History              38
Documentary          38
TV Movie              1
Name: count, dtype: int64


In [18]:
print(data_movies['country_main'].value_counts().head(20))

country_main
United States of America    4246
United Kingdom               504
France                       405
Canada                       350
Germany                      232
Japan                        129
India                        126
Australia                    125
China                        114
Belgium                      105
Spain                         84
Italy                         80
South Korea                   58
Hong Kong                     44
Denmark                       42
Russia                        42
Czech Republic                34
Ireland                       31
Mexico                        30
Brazil                        23
Name: count, dtype: int64


## 6. Categorías fuera de alcance y nulos
Elimino las películas para televisión, porque no tienen taquilla comparable, y las que no tienen país.

In [19]:
data_movies = data_movies[data_movies['genre_main'] != 'TV Movie']

In [20]:
print("genre_main nulos:", data_movies['genre_main'].isnull().sum())
print("country_main nulos:", data_movies['country_main'].isnull().sum())

genre_main nulos: 0
country_main nulos: 6


In [21]:
data_movies = data_movies.dropna(subset=['country_main'])
embudo.append(("Sin TV Movie ni país nulo", len(data_movies)))

In [22]:
print("Registros tras limpiar nulos:", len(data_movies))

Registros tras limpiar nulos: 7063


## 7. Duplicados
Reviso duplicados exactos y por título + año. No se encontró ninguno.

In [23]:
print("Duplicados exactos:", data_movies.duplicated().sum())
print("Títulos duplicados:", data_movies.duplicated(subset=['title', 'year']).sum())

Duplicados exactos: 0
Títulos duplicados: 0


In [24]:
# Ordeno por calificación para revisar el dataset de mejor a peor evaluada
data_movies = data_movies.sort_values('vote_average', ascending=False)

print("Registros finales:", len(data_movies))

Registros finales: 7063


## 8. Valores sospechosos
Algunas películas reportan ingresos o presupuestos de unos cuantos dólares (por ejemplo, 3 o 10),
lo que indica errores de captura (posiblemente cifras en millones). Los reviso a mano y elimino
los menores a $1,000.

In [25]:
print("=== vote_average ===")
print(data_movies['vote_average'].describe())
print("Fuera de rango [0-10]:", ((data_movies['vote_average'] < 0) | (data_movies['vote_average'] > 10)).sum())

print("\n=== revenue ===")
print(data_movies['revenue'].describe())
print("Revenue sospechoso (< 1000):", (data_movies['revenue'] < 1000).sum())

print("\n=== budget ===")
print(data_movies['budget'].describe())
print("Budget sospechoso (< 1000):", (data_movies['budget'] < 1000).sum())

=== vote_average ===
count    7063.000000
mean        6.540393
std         0.793777
min         1.908000
25%         6.031000
50%         6.568000
75%         7.106500
max         8.707000
Name: vote_average, dtype: float64
Fuera de rango [0-10]: 0

=== revenue ===
count    7.063000e+03
mean     9.164207e+07
std      1.788931e+08
min      1.000000e+00
25%      8.337282e+06
50%      3.005003e+07
75%      9.414469e+07
max      2.923706e+09
Name: revenue, dtype: float64
Revenue sospechoso (< 1000): 13

=== budget ===
count    7.063000e+03
mean     3.168019e+07
std      4.187769e+07
min      5.000000e+00
25%      6.000000e+06
50%      1.700000e+07
75%      4.000000e+07
max      4.600000e+08
Name: budget, dtype: float64
Budget sospechoso (< 1000): 12


In [26]:
print("=== Revenue sospechoso ===")
print(data_movies[data_movies['revenue'] < 1000][['title', 'year', 'revenue', 'budget']].to_string())

print("\n=== Budget sospechoso ===")
print(data_movies[data_movies['budget'] < 1000][['title', 'year', 'revenue', 'budget']].to_string())

=== Revenue sospechoso ===
                                              title  year  revenue    budget
15580                                  Thanks Boss!  2016        3    162297
12361                Chestnut: Hero of Central Park  2004       10         6
18015                                   Kaili Blues  2016      646        50
8219                                  Cross of Iron  1977      201   6000000
9485                                          Ben X  2007       27   1500000
16651                      Who Killed Captain Alex?  2010      200       200
16365                                    Trojan War  1997      309  15000000
7607                                    Housesitter  1992       94        26
10102                             Sex and Death 101  2007        1         5
15132                                   Hero Wanted  2008       10         6
9548   City Slickers II: The Legend of Curly's Gold  1994       43        40
5199       French Fried Vacation 3: Friends Forev

In [27]:
data_movies = data_movies[(data_movies['revenue'] >= 1000) & (data_movies['budget'] >= 1000)]
embudo.append(("Sin montos sospechosos (< $1,000)", len(data_movies)))
print("Registros finales:", len(data_movies))

Registros finales: 7046


In [28]:
print("Compañías únicas:", data_movies['production_companies'].nunique())
print("\nEjemplos:")
print(data_movies['production_companies'].head(10).to_string())

Compañías únicas: 6038

Ejemplos:
53                         Paramount, Alfran Productions
14                             Castle Rock Entertainment
215                       Paramount, The Coppola Company
119                                 Amblin Entertainment
991                                       Yash Raj Films
436               United Artists, Orion-Nova Productions
107                                        Studio Ghibli
81                                          Barunson E&A
2      DC Comics, Legendary Pictures, Syncopy, Isobel...
93     Castle Rock Entertainment, Darkwoods Productio...


## 9. Compañía productora
Hay más de 6,000 combinaciones de compañías. Tomo la principal (2,701 distintas) y conservo solo las
**23 con 30 o más películas**; el resto se agrupa como "Independiente". Con tantas categorías raras,
el modelo no podría aprender patrones confiables de cada una.

In [29]:
data_movies['company_main'] = data_movies['production_companies'].apply(
    lambda x: x.split(',')[0].strip() if pd.notnull(x) else None
)

print("Compañías únicas tras extraer primera:", data_movies['company_main'].nunique())
print("\nTop 20:")
print(data_movies['company_main'].value_counts().head(20))

Compañías únicas tras extraer primera: 2701

Top 20:
company_main
Paramount                    276
Universal Pictures           274
Columbia Pictures            192
Warner Bros. Pictures        137
New Line Cinema              126
Walt Disney Pictures         123
20th Century Fox              89
Touchstone Pictures           83
Metro-Goldwyn-Mayer           69
TriStar Pictures              63
United Artists                62
Miramax                       57
Lionsgate                     45
Orion Pictures                39
Village Roadshow Pictures     39
Dimension Films               39
Summit Entertainment          38
Amblin Entertainment          37
Screen Gems                   37
Fox Searchlight Pictures      35
Name: count, dtype: int64


In [30]:
# Compañías con más de 30 películas
top_companies = data_movies['company_main'].value_counts()
top_companies = top_companies[top_companies >= 30].index.tolist()

print("Compañías que conservamos:", len(top_companies))
print(top_companies)

Compañías que conservamos: 23
['Paramount', 'Universal Pictures', 'Columbia Pictures', 'Warner Bros. Pictures', 'New Line Cinema', 'Walt Disney Pictures', '20th Century Fox', 'Touchstone Pictures', 'Metro-Goldwyn-Mayer', 'TriStar Pictures', 'United Artists', 'Miramax', 'Lionsgate', 'Orion Pictures', 'Village Roadshow Pictures', 'Dimension Films', 'Summit Entertainment', 'Amblin Entertainment', 'Screen Gems', 'Fox Searchlight Pictures', 'DreamWorks Animation', 'Marvel Studios', 'Fox 2000 Pictures']


In [31]:
data_movies['company_main'] = data_movies['company_main'].apply(
    lambda x: x if x in top_companies else 'Independiente'
)

print("Distribución final:")
print(data_movies['company_main'].value_counts().head(10))
print("\nIndependiente:", (data_movies['company_main'] == 'Independiente').sum())

Distribución final:
company_main
Independiente            5089
Paramount                 276
Universal Pictures        274
Columbia Pictures         192
Warner Bros. Pictures     137
New Line Cinema           126
Walt Disney Pictures      123
20th Century Fox           89
Touchstone Pictures        83
Metro-Goldwyn-Mayer        69
Name: count, dtype: int64

Independiente: 5089


In [32]:
data_movies.head()

,id,title,vote_average,vote_count,revenue,budget,original_language,popularity,production_companies,year,genre_main,country_main,company_main
53,238,The Godfather,8.707,18677,245066411,6000000,en,158.448,"Paramount, Alfran Productions",1972,Drama,United States of America,Paramount
14,278,The Shawshank Redemption,8.702,24649,28341469,25000000,en,122.610,Castle Rock Entertainment,1994,Drama,United States of America,Independiente
215,240,The Godfather Part II,8.591,11293,102600000,13000000,en,64.753,"Paramount, The Coppola Company",1974,Drama,United States of America,Paramount
119,424,Schindler's List,8.573,14594,321365567,22000000,en,65.084,Amblin Entertainment,1993,Drama,United States of America,Amblin Entertainment
991,19404,Dilwale Dulhania Le Jayenge,8.552,4256,100000000,13200000,hi,33.575,Yash Raj Films,1995,Comedy,India,Independiente


In [33]:
data_movies = data_movies.drop(columns=['production_companies'])
print(data_movies.columns.tolist())

['id', 'title', 'vote_average', 'vote_count', 'revenue', 'budget', 'original_language', 'popularity', 'year', 'genre_main', 'country_main', 'company_main']


## 10. Transformación de popularidad
`popularity` está muy sesgada (pocas películas con valores enormes), así que aplico logaritmo
(`log1p`) para suavizar la distribución.

In [34]:
data_movies['popularity_log'] = np.log1p(data_movies['popularity'])
data_movies = data_movies.drop(columns=['popularity'])

print(data_movies.columns.tolist())

['id', 'title', 'vote_average', 'vote_count', 'revenue', 'budget', 'original_language', 'year', 'genre_main', 'country_main', 'company_main', 'popularity_log']


## 11. Variable objetivo: ¿éxito comercial?
Defino **éxito = ingreso mayor a la mediana** (≈ $30.2 millones de dólares). Así las clases quedan
balanceadas 50/50 (3,523 vs 3,523), lo que facilita entrenar y evaluar el modelo.

*Limitación:* los ingresos no están ajustados por inflación, así que las películas antiguas
tienden a quedar como "no exitosas". Lo retomo en las conclusiones del notebook 2.

In [35]:
mediana = data_movies['revenue'].median()
data_movies['target'] = (data_movies['revenue'] > mediana).astype(int)

print("Mediana de revenue:", mediana)
print("\nDistribución del target:")
print(data_movies['target'].value_counts())

Mediana de revenue: 30208746.0

Distribución del target:
target
1    3523
0    3523
Name: count, dtype: int64


## 12. Resumen: embudo de limpieza

In [36]:
tabla_embudo = pd.DataFrame(embudo, columns=["Paso", "Películas"])
tabla_embudo["Eliminadas"] = (-tabla_embudo["Películas"].diff()).fillna(0).astype(int)
tabla_embudo["% del original"] = (tabla_embudo["Películas"] / tabla_embudo["Películas"].iloc[0] * 100).round(2)
tabla_embudo

,Paso,Películas,Eliminadas,% del original
0,Dataset original de TMDB,994914,0,100.00
1,Con al menos 100 votos,18191,976723,1.83
2,Con año válido (≥ 1900),18177,14,1.83
3,Con ingreso y presupuesto > 0,7070,11107,0.71
4,Sin TV Movie ni país nulo,7063,7,0.71
5,"Sin montos sospechosos (< $1,000)",7046,17,0.71


In [37]:
data_movies.to_csv("../data/movies_clean.csv", index=False)
print("\n Guardado :)")


 Guardado :)


## Conclusión
De **994,914 registros** quedaron **7,046 películas** con datos completos y confiables, y una
variable objetivo balanceada. El dataset limpio se guarda en `data/movies_clean.csv`.